# GPU Experiment 5: Deep-Probing, Multi-Random Placebo & Drug-Disjoint Split Evaluation (Fixed Device & Dtype Match)

This notebook covers:
1. **20 Independent Random Vectors Distribution**: Evaluates $N=20$ isotropic random vectors ($v_{\text{rand}}^{(1..20)}$) to establish mean $\pm$ SD performance bounds.
2. **Vector Extraction Position Analysis**: Compares $v_{\text{steer}}$ extracted from `final_prompt_token`, `first_answer_token`, `final_answer_token`, and `mean_pooled`.
3. **Drug-Disjoint Train/Test Split**: Re-estimates $v_{\text{steer}}$ on non-overlapping drug entries to evaluate zero-shot generalizability.


In [1]:
!pip install -q rank_bm25 evaluate bert_score bitsandbytes accelerate transformers

import os, sys, json, time, math, torch
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import evaluate

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Count: {torch.cuda.device_count()}")
    print(f"GPU Device 0: {torch.cuda.get_device_name(0)}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 47.8 MB/s eta 0:00:00
PyTorch Version: 2.10.0+cu128
CUDA Available: True
GPU Count: 2
GPU Device 0: Tesla T4


In [2]:
possible_paths = [
    '/kaggle/input/datasets/anhemgithom/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/vietnamese-medical-halueval-15k/vietnamese_medical_halueval_15k_specialized.json',
    'e:/Paper_Steering_VN_15K/data/vietnamese_medical_halueval_15k_specialized.json',
    './data/vietnamese_medical_halueval_15k_specialized.json',
    './vietnamese_medical_halueval_15k_specialized.json'
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError("Dataset file not found!")

with open(data_path, 'r', encoding='utf-8') as f:
    full_dataset = json.load(f)

print(f"Total Dataset Size: {len(full_dataset)} records")
test_data = full_dataset[-500:]
train_data = full_dataset[:-2205]


Total Dataset Size: 14700 records


In [3]:
model_id = "Qwen/Qwen2.5-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.eval()
bertscore = evaluate.load("bertscore")
print("✅ Model and BERTScore initialized!")


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Model and BERTScore initialized!


In [4]:
def make_safe_hook(v_vector, alpha_0=18.0, K=16):
    step_counter = 0
    def hook_fn(module, input_tensor, output_tensor):
        nonlocal step_counter
        step_counter += 1
        if 1 <= step_counter <= K:
            alpha_t = alpha_0 * (1.0 - (step_counter - 1) / K)
            if isinstance(output_tensor, tuple):
                cur_tensor = output_tensor[0]
                v_curr = v_vector.to(device=cur_tensor.device, dtype=cur_tensor.dtype)
                modified = cur_tensor + alpha_t * v_curr
                return (modified,) + output_tensor[1:]
            else:
                v_curr = v_vector.to(device=output_tensor.device, dtype=output_tensor.dtype)
                return output_tensor + alpha_t * v_curr
        return output_tensor
    return hook_fn
print("✅ Device & Dtype safe hook function defined!")


✅ Device & Dtype safe hook function defined!


In [5]:
print("🚀 Running 20 Independent Random Vectors Distribution Experiment...")
hidden_dim = model.config.hidden_size
target_layer_module = model.model.layers[8]
eval_subset = test_data[:100]  # N=100 representative test questions

random_accs = []
torch.manual_seed(42)

for r_idx in range(20):
    v_rand_raw = torch.randn(hidden_dim)
    v_rand = v_rand_raw / v_rand_raw.norm(p=2)
    
    rand_gen, rand_refs, rand_hals = [], [], []
    for item in eval_subset:
        q_text = item['question']
        prompt = f"<|im_start|>user\n{q_text}<|im_end|>\n<|im_start|>assistant\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        prompt_len = inputs.input_ids.shape[1]
        
        hook_h = target_layer_module.register_forward_hook(make_safe_hook(v_rand))
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        hook_h.remove()
        
        gen_text = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)
        rand_gen.append(gen_text)
        rand_refs.append(item.get('right_answer', item.get('positive_answer')))
        rand_hals.append(item['hallucinated_answer'])
        
    r_bs_ref = bertscore.compute(predictions=rand_gen, references=rand_refs, model_type="bert-base-multilingual-cased")['f1']
    r_bs_hal = bertscore.compute(predictions=rand_gen, references=rand_hals, model_type="bert-base-multilingual-cased")['f1']
    r_acc = sum(1 for r, h in zip(r_bs_ref, r_bs_hal) if r > h) / len(eval_subset) * 100
    random_accs.append(r_acc)
    print(f"Random Vector {r_idx+1:02d}: Acc = {r_acc:.2f}%")

mean_rand_acc = np.mean(random_accs)
std_rand_acc = np.std(random_accs)
print(f"\n=== 20 RANDOM VECTORS SUMMARY ===")
print(f"Mean Accuracy: {mean_rand_acc:.2f}% ± {std_rand_acc:.2f}%")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🚀 Running 20 Independent Random Vectors Distribution Experiment...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Random Vector 01: Acc = 64.00%
Random Vector 02: Acc = 62.00%
Random Vector 03: Acc = 59.00%
Random Vector 04: Acc = 61.00%
Random Vector 05: Acc = 48.00%
Random Vector 06: Acc = 54.00%
Random Vector 07: Acc = 62.00%
Random Vector 08: Acc = 56.00%
Random Vector 09: Acc = 63.00%
Random Vector 10: Acc = 56.00%
Random Vector 11: Acc = 58.00%
Random Vector 12: Acc = 48.00%
Random Vector 13: Acc = 57.00%
Random Vector 14: Acc = 54.00%
Random Vector 15: Acc = 58.00%
Random Vector 16: Acc = 59.00%
Random Vector 17: Acc = 51.00%
Random Vector 18: Acc = 56.00%
Random Vector 19: Acc = 45.00%
Random Vector 20: Acc = 59.00%

=== 20 RANDOM VECTORS SUMMARY ===
Mean Accuracy: 56.50% ± 5.11%


In [6]:
print("🚀 Running Vector Extraction Position Probing...")

def extract_vector_at_position(position_type="final_answer_token", layer_idx=8):
    pos_acts, neg_acts = [], []
    for item in train_data[:150]:
        q = item['question']
        pos_ans = item.get('right_answer', item.get('positive_answer'))
        neg_ans = item['hallucinated_answer']
        
        prompt_pos = f"<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{pos_ans}"
        prompt_neg = f"<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{neg_ans}"
        
        with torch.no_grad():
            inp_p = tokenizer(prompt_pos, return_tensors="pt").to(model.device)
            out_p = model(inp_p.input_ids, output_hidden_states=True)
            states_p = out_p.hidden_states[layer_idx][0]
            
            inp_n = tokenizer(prompt_neg, return_tensors="pt").to(model.device)
            out_n = model(inp_n.input_ids, output_hidden_states=True)
            states_n = out_n.hidden_states[layer_idx][0]
            
            if position_type == "final_prompt_token":
                pos_acts.append(states_p[len(tokenizer(q).input_ids)-1, :].detach().cpu())
                neg_acts.append(states_n[len(tokenizer(q).input_ids)-1, :].detach().cpu())
            elif position_type == "first_answer_token":
                pos_acts.append(states_p[-len(tokenizer(pos_ans).input_ids), :].detach().cpu())
                neg_acts.append(states_n[-len(tokenizer(neg_ans).input_ids), :].detach().cpu())
            elif position_type == "mean_pooled":
                pos_acts.append(states_p.mean(dim=0).detach().cpu())
                neg_acts.append(states_n.mean(dim=0).detach().cpu())
            else:  # final_answer_token
                pos_acts.append(states_p[-1, :].detach().cpu())
                neg_acts.append(states_n[-1, :].detach().cpu())
                
    v_diff = torch.stack(pos_acts).mean(dim=0) - torch.stack(neg_acts).mean(dim=0)
    return v_diff / v_diff.norm(p=2)

positions = ["final_prompt_token", "first_answer_token", "final_answer_token", "mean_pooled"]
position_results = {}

for pos in positions:
    print(f"Evaluating Extraction Position: {pos}...")
    v_pos = extract_vector_at_position(pos)
    
    gen_list, ref_list, hal_list = [], [], []
    for item in test_data[:100]:
        prompt = f"<|im_start|>user\n{item['question']}<|im_end|>\n<|im_start|>assistant\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        p_len = inputs.input_ids.shape[1]
        
        hook_h = target_layer_module.register_forward_hook(make_safe_hook(v_pos))
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        hook_h.remove()
        
        gen_list.append(tokenizer.decode(out[0][p_len:], skip_special_tokens=True))
        ref_list.append(item.get('right_answer', item.get('positive_answer')))
        hal_list.append(item['hallucinated_answer'])
        
    bs_ref = bertscore.compute(predictions=gen_list, references=ref_list, model_type="bert-base-multilingual-cased")['f1']
    bs_hal = bertscore.compute(predictions=gen_list, references=hal_list, model_type="bert-base-multilingual-cased")['f1']
    acc = sum(1 for r, h in zip(bs_ref, bs_hal) if r > h) / 100.0 * 100
    position_results[pos] = {"accuracy": acc, "bertscore": float(np.mean(bs_ref))}
    print(f"Position {pos:20s}: Accuracy = {acc:.2f}%, BERTScore = {np.mean(bs_ref):.4f}")


🚀 Running Vector Extraction Position Probing...
Evaluating Extraction Position: final_prompt_token...
Position final_prompt_token  : Accuracy = 51.00%, BERTScore = 0.7010
Evaluating Extraction Position: first_answer_token...
Position first_answer_token  : Accuracy = 50.00%, BERTScore = 0.6990
Evaluating Extraction Position: final_answer_token...
Position final_answer_token  : Accuracy = 55.00%, BERTScore = 0.7014
Evaluating Extraction Position: mean_pooled...
Position mean_pooled         : Accuracy = 59.00%, BERTScore = 0.6995


In [7]:
exp5_summary = {
    "random_vectors_distribution": {
        "num_vectors": 20,
        "mean_acc": float(mean_rand_acc),
        "std_acc": float(std_rand_acc),
        "all_accs": random_accs
    },
    "position_probing": position_results
}

with open("deep_probing_and_extraction_results.json", "w", encoding="utf-8") as f:
    json.dump(exp5_summary, f, indent=2, ensure_ascii=False)

print("✅ Saved deep_probing_and_extraction_results.json successfully!")
print(json.dumps(exp5_summary, indent=2))


✅ Saved deep_probing_and_extraction_results.json successfully!
{
  "random_vectors_distribution": {
    "num_vectors": 20,
    "mean_acc": 56.5,
    "std_acc": 5.113707070218238,
    "all_accs": [
      64.0,
      62.0,
      59.0,
      61.0,
      48.0,
      54.0,
      62.0,
      56.00000000000001,
      63.0,
      56.00000000000001,
      57.99999999999999,
      48.0,
      56.99999999999999,
      54.0,
      57.99999999999999,
      59.0,
      51.0,
      56.00000000000001,
      45.0,
      59.0
    ]
  },
  "position_probing": {
    "final_prompt_token": {
      "accuracy": 51.0,
      "bertscore": 0.7010485017299652
    },
    "first_answer_token": {
      "accuracy": 50.0,
      "bertscore": 0.6989830684661865
    },
    "final_answer_token": {
      "accuracy": 55.00000000000001,
      "bertscore": 0.7014082396030425
    },
    "mean_pooled": {
      "accuracy": 59.0,
      "bertscore": 0.6994546562433243
    }
  }
}
